# Camada Bronze - Electronic Products and Pricing (Datafiniti) + Amazon Review Data 2018 (Quévin)

Carrega os ficheiros originais para o HDFS e cria as tabelas Bronze em formato Delta, na stack comum do grupo
(base `bronze`, caminho `/projeto/bronze/quevin/`), com os metadados de ingestão `_data_ingestao` e `_ficheiro_origem`.

| Dataset | Ficheiro | Tabela Bronze |
|---|---|---|
| Electronic Products and Pricing Data (principal) | DatafinitiElectronicsProductsPricingData.csv | bronze.datafiniti_precos |
| Amazon Review Data 2018, Electronics (complementar) | Electronics.csv (avaliações, sem cabeçalho) | bronze.amazon_ratings |
| Amazon Review Data 2018, Electronics (complementar) | meta_Electronics.json.gz (produtos) | bronze.amazon_meta |

Fontes: Kaggle `datafiniti/electronic-products-prices`; Amazon Review Data 2018 (UCSD, J. McAuley), ficheiros
`categoryFilesSmall/Electronics.csv` e `metaFiles2/meta_Electronics.json.gz`.

- A Datafiniti e as avaliações ficam com todas as colunas, como texto. O CSV das avaliações não tem cabeçalho:
  os nomes são os da documentação (`asin`, `reviewerID`, `overall`, `unixReviewTime`).
- O ficheiro de produtos tem cerca de 11 GB descomprimido e 19 campos, a maioria listas e texto livre
  (imagens, descrições, `also_buy`...). O ficheiro original completo fica no HDFS; a tabela Bronze guarda,
  tal como vêm, os 8 campos que a Silver usa.

In [ ]:
ficheiros = {
    "/home/jovyan/work/data/quevin/DatafinitiElectronicsProductsPricingData.csv": "/projeto/bronze/quevin/DatafinitiElectronicsProductsPricingData.csv",
    "/home/jovyan/work/data/quevin/Electronics.csv": "/projeto/bronze/quevin/Electronics.csv",
    "/home/jovyan/work/data/quevin/meta_Electronics.json.gz": "/projeto/bronze/quevin/meta_Electronics.json.gz",
}

In [ ]:
# Copiar os ficheiros originais para a zona Bronze do HDFS (full load: apaga a versão anterior e volta a carregar)
from hdfs import InsecureClient
client = InsecureClient("http://hdfs-nn:9870", user="anonymous")

for from_path, to_path in ficheiros.items():
    client.delete(to_path)
    client.upload(to_path, from_path)
    print(to_path, client.status(to_path)["length"], "bytes")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta import *

# Stack comum do grupo: HDFS (hdfs-nn), Hive Metastore e Delta Lake 2.4
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

builder = SparkSession \
    .builder \
    .appName("Bronze - Datafiniti + Amazon (Quévin)") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.session.timeZone", "UTC") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Os nomes originais das colunas têm espaços e caracteres especiais;
# o column mapping do Delta permite guardá-los tal como vêm da fonte
spark.conf.set("spark.databricks.delta.properties.defaults.columnMapping.mode", "name")
spark.conf.set("spark.databricks.delta.properties.defaults.minReaderVersion", "2")
spark.conf.set("spark.databricks.delta.properties.defaults.minWriterVersion", "5")

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze LOCATION 'hdfs://hdfs-nn:9000/warehouse/bronze.db/'")

def gravar_bronze(df, tabela):
    """Acrescenta os metadados de ingestão e grava a tabela Delta tal como veio da fonte."""
    df = df.withColumn("_data_ingestao", F.current_timestamp()) \
           .withColumn("_ficheiro_origem", F.input_file_name())
    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .option("path", f"hdfs://hdfs-nn:9000/warehouse/bronze.db/{tabela}/") \
        .saveAsTable(f"bronze.{tabela}")
    print(f"bronze.{tabela}: {spark.table(f'bronze.{tabela}').count()} linhas, {len(df.columns) - 2} colunas originais")

def ler_csv(hdfs_path, **opcoes):
    """Lê um CSV do HDFS com todas as colunas como texto (sem inferSchema)."""
    leitor = spark.read.option("header", "true").option("multiLine", "true").option("escape", '"')
    for k, v in opcoes.items():
        leitor = leitor.option(k, v)
    return leitor.csv("hdfs://hdfs-nn:9000" + hdfs_path)

In [ ]:
# Datafiniti: o cabeçalho tem 5 colunas vazias no fim (vírgulas a mais); ficam com o nome que o Spark lhes dá (_c26 ...)
gravar_bronze(ler_csv("/projeto/bronze/quevin/DatafinitiElectronicsProductsPricingData.csv"), "datafiniti_precos")

# Avaliações da Amazon: CSV sem cabeçalho e sem campos com aspas/mudanças de linha
ratings = spark.read.csv("hdfs://hdfs-nn:9000/projeto/bronze/quevin/Electronics.csv", header=False,
                         schema="asin STRING, reviewerID STRING, overall STRING, unixReviewTime STRING")
gravar_bronze(ratings, "amazon_ratings")

In [ ]:
# Produtos da Amazon: o .gz não se divide em partes, por isso é lido por uma só tarefa e redistribuído antes de gravar
meta = spark.read.json("hdfs://hdfs-nn:9000/projeto/bronze/quevin/meta_Electronics.json.gz",
                       schema="asin STRING, title STRING, brand STRING, main_cat STRING, category ARRAY<STRING>, "
                              "date STRING, price STRING, rank STRING")
gravar_bronze(meta.repartition(24), "amazon_meta")

## Verificação

As tabelas Bronze têm de ter o mesmo número de registos dos ficheiros originais.

In [ ]:
import csv, gzip
with open("/home/jovyan/work/data/quevin/DatafinitiElectronicsProductsPricingData.csv", encoding="utf-8", newline="") as f:
    n_dat = sum(1 for _ in csv.reader(f)) - 1
with open("/home/jovyan/work/data/quevin/Electronics.csv", "rb") as f:
    n_rat = sum(1 for _ in f)
with gzip.open("/home/jovyan/work/data/quevin/meta_Electronics.json.gz", "rb") as f:
    n_meta = sum(1 for _ in f)

for tabela, n in {"datafiniti_precos": n_dat, "amazon_ratings": n_rat, "amazon_meta": n_meta}.items():
    real = spark.table(f"bronze.{tabela}").count()
    print(f"{tabela:18s} ficheiro {n:>9}  bronze {real:>9}  {'OK' if n == real else 'DIFERENTE'}")
    assert n == real

In [ ]:
spark.table("bronze.amazon_ratings").show(3)
spark.table("bronze.amazon_meta").select("asin", "title", "brand", "main_cat", "date", "price").show(3, 40)

In [ ]:
spark.stop()